# Part A: Numerical Prediction - Step 5: Ensemble Models

This notebook builds and evaluates regression ensemble models (`VotingRegressor` and `StackingRegressor`) on the CarDekho car price dataset using the exact same train/test split (`test_size=0.2`, `random_state=42`) and preprocessing pipeline defined in `01_eda_and_preprocessing.ipynb`.

We compare individual base regressors against the ensemble models on the same held-out test set (`X_test_processed`, `y_test`) using **MAE**, **RMSE**, and **$R^2$**.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor, StackingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Load Data & Apply Consistent Preprocessing

To guarantee a fair comparison across all Part A notebooks, we load `../data/processed/numerical_split.joblib` if it exists; otherwise, we reproduce the exact deduplication, feature engineering (`brand`, `car_age`), and `train_test_split(..., test_size=0.2, random_state=42)` from `01_eda_and_preprocessing.ipynb`.

In [ ]:
split_path = "../data/processed/numerical_split.joblib"
raw_data_path = "../data/CAR DETAILS FROM CAR DEKHO.csv"

if os.path.exists(split_path):
    X_train, X_test, y_train, y_test = joblib.load(split_path)
    print("Loaded pre-saved train/test split from numerical_split.joblib.")
else:
    df = pd.read_csv(raw_data_path)
    df_clean = df.drop_duplicates().copy()
    df_clean["brand"] = df_clean["name"].apply(lambda x: str(x).split()[0])
    df_clean["car_age"] = 2020 - df_clean["year"]

    X = df_clean.drop(columns=["selling_price"])
    y = df_clean["selling_price"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    print("Created train/test split directly from raw CSV with random_state=42.")

numerical_features = ["year", "car_age", "km_driven"]
categorical_features = ["fuel", "seller_type", "transmission", "owner", "brand"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
).set_output(transform="pandas")

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train_processed: {X_train_processed.shape}, X_test_processed: {X_test_processed.shape}")

## 2. Train Base Regression Models

We train four diverse base models on `X_train_processed`, `y_train`:
- **Linear Regression** (unregularized linear baseline)
- **Ridge Regression** ($L_2$-regularized linear model, handles one-hot brand multicollinearity)
- **KNN Regressor** (non-parametric instance-based learner)
- **Random Forest Regressor** (non-linear tree-based ensemble)

In [ ]:
base_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "KNN Regression (k=5)": KNeighborsRegressor(n_neighbors=5, weights="distance"),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
}

def evaluate_regressor(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    mae = mean_absolute_error(y_te, preds)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    r2 = r2_score(y_te, preds)
    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}, preds

results = []
predictions = {}

for name, model in base_models.items():
    metrics, preds = evaluate_regressor(name, model, X_train_processed, y_train, X_test_processed, y_test)
    results.append(metrics)
    predictions[name] = preds
    print(f"{name:25s} | MAE: {metrics['MAE']:,.2f} | RMSE: {metrics['RMSE']:,.2f} | R2: {metrics['R2']:.4f}")

## 3. Build & Evaluate Ensemble Models

We construct two ensemble architectures combining complementary base learners (`Ridge`, `KNeighborsRegressor`, and `RandomForestRegressor`):
1. **VotingRegressor**: Averages the predictions of the diverse base regressors (with higher weight on stronger non-linear learners or equal weights).
2. **StackingRegressor**: Trains a `Ridge` meta-regressor using 5-fold cross-validated out-of-fold predictions from the base estimators.

In [ ]:
voting_reg = VotingRegressor(
    estimators=[
        ("ridge", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
        ("knn", KNeighborsRegressor(n_neighbors=5, weights="distance")),
        ("rf", RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
    ],
    weights=[1, 1, 2],
)

stacking_reg = StackingRegressor(
    estimators=[
        ("ridge", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
        ("knn", KNeighborsRegressor(n_neighbors=5, weights="distance")),
        ("rf", RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
    ],
    final_estimator=Ridge(alpha=1.0, random_state=RANDOM_STATE),
    cv=5,
    n_jobs=-1,
)

for name, model in [("Voting Regressor", voting_reg), ("Stacking Regressor", stacking_reg)]:
    metrics, preds = evaluate_regressor(name, model, X_train_processed, y_train, X_test_processed, y_test)
    results.append(metrics)
    predictions[name] = preds
    print(f"{name:25s} | MAE: {metrics['MAE']:,.2f} | RMSE: {metrics['RMSE']:,.2f} | R2: {metrics['R2']:.4f}")

## 4. Model Comparison Table & Visualizations

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="RMSE").reset_index(drop=True)
display(results_df.style.format({"MAE": "{:,.2f}", "RMSE": "{:,.2f}", "R2": "{:.4f}"}))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(data=results_df, x="MAE", y="Model", ax=axes[0], palette="Blues_r", hue="Model", legend=False)
axes[0].set_title("MAE Comparison (Lower is Better)")

sns.barplot(data=results_df, x="RMSE", y="Model", ax=axes[1], palette="Greens_r", hue="Model", legend=False)
axes[1].set_title("RMSE Comparison (Lower is Better)")

sns.barplot(data=results_df, x="R2", y="Model", ax=axes[2], palette="Purples", hue="Model", legend=False)
axes[2].set_title("R2 Score Comparison (Higher is Better)")

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs. Predicted scatter plot for top base model vs. Stacking Regressor
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model_name, color in [
    (axes[0], "Random Forest", "teal"),
    (axes[1], "Stacking Regressor", "darkorange"),
]:
    y_pred = predictions[model_name]
    ax.scatter(y_test, y_pred, alpha=0.5, color=color)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, "r--", lw=2)
    ax.set_title(f"{model_name}: Actual vs. Predicted")
    ax.set_xlabel("Actual Selling Price")
    ax.set_ylabel("Predicted Selling Price")

plt.tight_layout()
plt.show()

## 5. Discussion: How the Ensemble Works & Performance Analysis

### How the Ensembles Work
- **VotingRegressor** combines predictions from heterogeneous base models (linear `Ridge`, instance-based `KNN`, and tree-based `RandomForest`) by computing a weighted average. Because each model family makes different inductive assumptions, uncorrelated prediction errors tend to cancel out, reducing overall variance.
- **StackingRegressor** goes one step further by generating 5-fold out-of-fold predictions on `X_train_processed` from each base estimator and training a `Ridge` meta-learner on those predictions. Instead of fixed manual weights, the meta-learner learns optimal weights (and potential bias corrections) for combining the base models.

### Why Performance Improved (or Did Not Improve) vs. Base Models
- **Complementarity of Model Families:** Linear models (`LinearRegression` / `Ridge`) capture global additive trends (e.g., depreciation with `car_age` and brand premiums) but miss non-linear interactions, whereas `KNN` and `RandomForest` capture local neighborhoods and non-linear feature interactions (e.g., luxury brands depreciating differently by transmission/fuel type).
- **When Stacking Outperforms Voting & Base Models:** `StackingRegressor` learns to trust the stronger non-linear estimators (`RandomForest` / `KNN`) while using `Ridge` to stabilize extrapolations on rare brands or extreme mileages, often achieving lower RMSE and higher $R^2$ than any single base model.
- **When Voting May Underperform the Best Base Model:** If a weaker model (such as `Ridge` or un-tuned `KNN`) has substantially higher error on skewed car prices, fixed averaging in `VotingRegressor` can pull the combined prediction away from the stronger `RandomForest` predictions. In such cases, `StackingRegressor` or tuning weights is preferred.

## 6. Save Best Ensemble Model

In [ ]:
os.makedirs("../models", exist_ok=True)
joblib.dump(stacking_reg, "../models/numerical_ensemble_model.joblib")
print("Saved Stacking Regressor to ../models/numerical_ensemble_model.joblib")